In [ ]:
# instalar dependências

!pip install -r requirements.txt

In [ ]:
# imports

# módulos do Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import cross_val_score, StratifiedKFold, LeaveOneOut
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, f1_score)
from sklearn.preprocessing import LabelEncoder

# módulos de matemática e geração de gráficos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# módulos padrões de python e de sistema
import warnings
import os
from math import pi

In [ ]:
#configurações do script

warnings.filterwarnings('ignore')
matplotlib.use('Agg')


# Configuração visual
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

OUTPUT_DIR = "./resultados"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# importando e lendo o dataset

df = pd.read_csv("dataset.csv")

In [ ]:
# Apenas como referência, calcular o ICE Score das oportunidades

df['ice_score'] = df['impacto'] * df['confianca'] * df['facilidade']

In [ ]:
# Classificação baseada no score ICE com quartis
# Alta, Média, Baixa prioridade

def classificar_prioridade(score, q33, q66):
    if score >= q66:
        return 'Alta'
    elif score >= q33:
        return 'Média'
    else:
        return 'Baixa'

q33 = df['ice_score'].quantile(0.33)
q66 = df['ice_score'].quantile(0.66)

df['prioridade'] = df['ice_score'].apply(lambda x: classificar_prioridade(x, q33, q66))

# Encoding do perfil do time para o modelo
perfil_map = {'Júnior': 0, 'Pleno': 1, 'Sênior': 2}
df['perfil_time_enc'] = df['perfil_time'].map(perfil_map)

print("*" * 20)
print("Informações do Dataset")
print("*" * 20)
print(f"\nTotal de oportunidades: {len(df)}")
print(f"Cenários: {df['cenario'].nunique()}")
print(f"Setores: {df['setor'].nunique()}")
print(f"\nDistribuição das classes de prioridade:")
print(df['prioridade'].value_counts())
print(f"\nQuartis ICE: Q33={q33:.0f}, Q66={q66:.0f}")
print(f"\nICE Score - Min: {df['ice_score'].min()}, Max: {df['ice_score'].max()}, "
      f"Média: {df['ice_score'].mean():.1f}")

df.to_csv(f'{OUTPUT_DIR}/dataset_completo.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Features: Impacto, Confiança, Facilidade, Perfil do Time, Tamanho do Time
FEATURES = ['impacto', 'confianca', 'facilidade', 'perfil_time_enc', 'tamanho_time']
X = df[FEATURES]
y = df['prioridade']

le = LabelEncoder()
y_enc = le.fit_transform(y)

print("\n" + "*" * 20)
print("Realizando o treinamento dos modelos")
print("*" * 20)

# --- Modelo 1: Random Forest ---
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=3,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

# --- Modelo 2: Árvore de Decisão Única (comparação) ---
dt = DecisionTreeClassifier(
    max_depth=3,
    min_samples_split=3,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

In [ ]:
# --- Cross-validation com Leave-One-Out (ideal para N pequeno) ---
loo = LeaveOneOut()

# Random Forest
rf_scores = cross_val_score(rf, X, y_enc, cv=loo, scoring='accuracy')
rf_f1_scores = cross_val_score(rf, X, y_enc, cv=loo, scoring='f1_weighted')

# Decision Tree
dt_scores = cross_val_score(dt, X, y_enc, cv=loo, scoring='accuracy')
dt_f1_scores = cross_val_score(dt, X, y_enc, cv=loo, scoring='f1_weighted')

print(f"\n--- Leave-One-Out Cross-Validation (n={len(df)}) ---")
print(f"\nRandom Forest:")
print(f"  Acurácia: {rf_scores.mean():.3f} (±{rf_scores.std():.3f})")
print(f"  F1-Score (weighted): {rf_f1_scores.mean():.3f} (±{rf_f1_scores.std():.3f})")
print(f"\nÁrvore de Decisão Única:")
print(f"  Acurácia: {dt_scores.mean():.3f} (±{dt_scores.std():.3f})")
print(f"  F1-Score (weighted): {dt_f1_scores.mean():.3f} (±{dt_f1_scores.std():.3f})")


In [ ]:
# Stratified K-Fold também (para fins de estudo)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_skf_scores = cross_val_score(rf, X, y_enc, cv=skf, scoring='accuracy')
dt_skf_scores = cross_val_score(dt, X, y_enc, cv=skf, scoring='accuracy')

print(f"\n--- Stratified 5-Fold Cross-Validation ---")
print(f"\nRandom Forest:")
print(f"  Acurácia: {rf_skf_scores.mean():.3f} (±{rf_skf_scores.std():.3f})")
print(f"\nÁrvore de Decisão Única:")
print(f"  Acurácia: {dt_skf_scores.mean():.3f} (±{dt_skf_scores.std():.3f})")

In [ ]:
# Treinar modelos finais no dataset completo (para feature importance e visualizações)
rf.fit(X, y_enc)
dt.fit(X, y_enc)

# Predições no dataset completo (para matriz de confusão ilustrativa)
y_pred_rf = rf.predict(X)
y_pred_dt = dt.predict(X)

print(f"\n--- Relatório de Classificação (Random Forest - treino completo) ---")
print(classification_report(y_enc, y_pred_rf, target_names=le.classes_))

print(f"\n--- Relatório de Classificação (Árvore de Decisão - treino completo) ---")
print(classification_report(y_enc, y_pred_dt, target_names=le.classes_))


In [ ]:

# Paleta de cores
COLORS = {'Alta': '#2ecc71', 'Média': '#f39c12', 'Baixa': '#e74c3c'}
PALETTE = ['#e74c3c', '#f39c12', '#2ecc71']  # Baixa, Média, Alta

# --- 4.1 Feature Importance (Random Forest) ---
fig, ax = plt.subplots(figsize=(10, 6))
importance = pd.Series(rf.feature_importances_, index=['Impacto', 'Confiança', 
                        'Facilidade', 'Senioridade\ndo Time', 'Tamanho\ndo Time'])
importance = importance.sort_values(ascending=True)
bars = ax.barh(importance.index, importance.values, color='#3498db', edgecolor='white')
ax.set_xlabel('Importância Relativa', fontsize=12)
ax.set_title('Importância das Features — Random Forest', fontsize=14, fontweight='bold')
for bar, val in zip(bars, importance.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', 
            va='center', fontsize=11)
ax.set_xlim(0, importance.max() * 1.15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_feature_importance.png', bbox_inches='tight')
plt.close()
print("\n✓ Gráfico 1: Feature Importance salvo")

# --- 4.2 Matriz de Confusão (ambos modelos lado a lado) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(axes, [y_pred_rf, y_pred_dt], 
                               ['Random Forest', 'Árvore de Decisão Única']):
    cm = confusion_matrix(y_enc, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_,
                cbar_kws={'shrink': 0.8})
    ax.set_xlabel('Predito', fontsize=12)
    ax.set_ylabel('Real', fontsize=12)
    ax.set_title(f'Matriz de Confusão — {title}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_matrizes_confusao.png', bbox_inches='tight')
plt.close()
print("✓ Gráfico 2: Matrizes de Confusão salvo")

# --- 4.3 Matriz de Priorização (Impacto × Facilidade, cor = Prioridade) ---
fig, ax = plt.subplots(figsize=(12, 8))

for prioridade, color in COLORS.items():
    mask = df['prioridade'] == prioridade
    scatter = ax.scatter(df.loc[mask, 'facilidade'], df.loc[mask, 'impacto'],
                        c=color, s=df.loc[mask, 'confianca'] * 25,
                        alpha=0.7, edgecolors='white', linewidth=1.5,
                        label=f'{prioridade} Prioridade', zorder=3)

# Anotar pontos com nome da oportunidade
for _, row in df.iterrows():
    ax.annotate(row['oportunidade'], (row['facilidade'], row['impacto']),
               fontsize=6, alpha=0.8, ha='center', va='bottom',
               xytext=(0, 8), textcoords='offset points')

ax.set_xlabel('Facilidade', fontsize=13)
ax.set_ylabel('Impacto', fontsize=13)
ax.set_title('Matriz de Priorização — Impacto × Facilidade\n(tamanho do ponto = Confiança)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim(1, 10.5)
ax.set_ylim(1, 10.5)
ax.axhline(y=7, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=6, color='gray', linestyle='--', alpha=0.3)

# Quadrantes
ax.text(8.5, 9.5, 'PRIORIZAR\n(Alto Impacto +\nAlta Facilidade)', 
        ha='center', fontsize=9, color='#27ae60', fontweight='bold', alpha=0.6)
ax.text(3, 9.5, 'PLANEJAR\n(Alto Impacto +\nBaixa Facilidade)', 
        ha='center', fontsize=9, color='#f39c12', fontweight='bold', alpha=0.6)
ax.text(3, 2, 'DESCARTAR\n(Baixo Impacto +\nBaixa Facilidade)', 
        ha='center', fontsize=9, color='#e74c3c', fontweight='bold', alpha=0.6)
ax.text(8.5, 2, 'QUICK WINS\n(Baixo Impacto +\nAlta Facilidade)', 
        ha='center', fontsize=9, color='#3498db', fontweight='bold', alpha=0.6)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_matriz_priorizacao.png', bbox_inches='tight')
plt.close()
print("✓ Gráfico 3: Matriz de Priorização salvo")

# --- 4.4 Heatmap Comparativo ICE por Cenário ---
# Pivot: média dos scores ICE por cenário
pivot_data = df.groupby('cenario')[['impacto', 'confianca', 'facilidade']].mean()
pivot_data.columns = ['Impacto', 'Confiança', 'Facilidade']

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=1, linecolor='white', cbar_kws={'label': 'Média do Score'})
ax.set_title('Média dos Indicadores ICE por Cenário', fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_heatmap_cenarios.png', bbox_inches='tight')
plt.close()
print("✓ Gráfico 4: Heatmap ICE por Cenário salvo")

# --- 4.5 Comparação MVNO: Time Pleno vs Sênior ---
mvno_comp = df[df['cenario'] == 'MVNO'].copy()
mvno_pivot = mvno_comp.pivot_table(
    index='oportunidade', 
    columns='perfil_time', 
    values=['impacto', 'confianca', 'facilidade']
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
criterios = ['impacto', 'confianca', 'facilidade']
titulos = ['Impacto', 'Confiança', 'Facilidade']
colors_comp = ['#3498db', '#e67e22']

for ax, criterio, titulo in zip(axes, criterios, titulos):
    data = mvno_pivot[criterio]
    data.plot(kind='barh', ax=ax, color=colors_comp, edgecolor='white', width=0.7)
    ax.set_title(f'{titulo}', fontsize=19, fontweight='bold')
    ax.set_xlabel('Score (1-10)', fontsize=15)
    ax.set_ylabel('')
    ax.legend(title='Perfil do Time', fontsize=9)
    ax.set_xlim(0, 11)

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')

fig.suptitle('Comparação MVNO: Time Pleno vs. Sênior', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_comparacao_mvno.png', bbox_inches='tight')
plt.close()
print("✓ Gráfico 5: Comparação MVNO Pleno vs Sênior salvo")

# --- 4.6 Distribuição das Classes de Prioridade por Cenário ---
fig, ax = plt.subplots(figsize=(12, 6))
ct = pd.crosstab(df['cenario'], df['prioridade'])
ct = ct[['Baixa', 'Média', 'Alta']]  # ordenar
ct.plot(kind='bar', stacked=True, ax=ax, color=PALETTE, edgecolor='white', width=0.7)
ax.set_title('Distribuição das Classes de Prioridade por Cenário', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Número de Oportunidades', fontsize=12)
ax.legend(title='Prioridade', fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_distribuicao_prioridade_cenario.png', bbox_inches='tight')
plt.close()
print("✓ Gráfico 6: Distribuição de Prioridade por Cenário salvo")

# --- 4.7 Comparação de Performance: RF vs DT ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.grid(False)
metrics_comp = pd.DataFrame({
    'Modelo': ['Random Forest', 'Random Forest', 'Árvore de Decisão', 'Árvore de Decisão'],
    'Métrica': ['Acurácia (LOO)', 'F1-Score (LOO)', 'Acurácia (LOO)', 'F1-Score (LOO)'],
    'Valor': [rf_scores.mean(), rf_f1_scores.mean(), dt_scores.mean(), dt_f1_scores.mean()]
})

bars = sns.barplot(data=metrics_comp, x='Métrica', y='Valor', hue='Modelo', 
                   palette=['#3498db', '#e67e22'], ax=ax, edgecolor='white')
ax.set_ylim(0, 1.0)
ax.set_title('Comparação de Performance: Random Forest vs. Árvore de Decisão\n(Leave-One-Out Cross-Validation)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('')

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=10, padding=3)

ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_comparacao_modelos.png', bbox_inches='tight')
plt.close()
print("✓ Gráfico 7: Comparação de Modelos salvo")

# --- 4.8 Radar Chart por Cenário (média ICE) ---

categorias = ['Impacto', 'Confiança', 'Facilidade']
N = len(categorias)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cenarios_radar = df.groupby('cenario')[['impacto', 'confianca', 'facilidade']].mean()

colors_radar = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c', '#e67e22']

for idx, (cenario, values) in enumerate(cenarios_radar.iterrows()):
    vals = values.tolist()
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=cenario, 
            color=colors_radar[idx % len(colors_radar)])
    ax.fill(angles, vals, alpha=0.08, color=colors_radar[idx % len(colors_radar)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categorias, fontsize=12)
ax.set_ylim(0, 10)
ax.set_title('Perfil ICE Médio por Cenário', fontsize=14, fontweight='bold', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_radar_cenarios.png', bbox_inches='tight')
plt.close()
print("✓ Gráfico 8: Radar Chart por Cenário salvo")

# --- 4.9 Árvore de Decisão (interpretabilidade) ---
tree_rules = export_text(dt, feature_names=['Impacto', 'Confiança', 'Facilidade', 
                                             'Senioridade', 'Tamanho Time'],
                         class_names=list(le.classes_))
print(f"\n--- Regras da Árvore de Decisão ---")
print(tree_rules)

# Salvar regras
with open(f'{OUTPUT_DIR}/arvore_decisao_regras.txt', 'w') as f:
    f.write("Regras da Árvore de Decisão\n")
    f.write("=" * 50 + "\n\n")
    f.write(tree_rules)


In [ ]:
print("\n" + "*" * 20)
print("RESUMO DO EXPERIMENTO")
print("=" * 20)
print(f"""
Dataset:
  - {len(df)} oportunidades em {df['cenario'].nunique()} cenários ({df['setor'].nunique()} setores)
  - 3 classes de prioridade: Alta ({(df['prioridade']=='Alta').sum()}), 
    Média ({(df['prioridade']=='Média').sum()}), Baixa ({(df['prioridade']=='Baixa').sum()})
  - Features: Impacto, Confiança, Facilidade, Senioridade do Time, Tamanho do Time

Random Forest (n_estimators=100, max_depth=5):
  - LOO Acurácia: {rf_scores.mean():.3f} (±{rf_scores.std():.3f})
  - LOO F1-Score: {rf_f1_scores.mean():.3f} (±{rf_f1_scores.std():.3f})
  - 5-Fold Acurácia: {rf_skf_scores.mean():.3f} (±{rf_skf_scores.std():.3f})
  - Top features: {', '.join(importance.sort_values(ascending=False).index[:3])}

Árvore de Decisão Única (max_depth=3):
  - LOO Acurácia: {dt_scores.mean():.3f} (±{dt_scores.std():.3f})
  - LOO F1-Score: {dt_f1_scores.mean():.3f} (±{dt_f1_scores.std():.3f})
  - 5-Fold Acurácia: {dt_skf_scores.mean():.3f} (±{dt_skf_scores.std():.3f})

Gráficos gerados: 8 visualizações + regras da árvore
""")

print(f"\nTodos os arquivos salvos em: {OUTPUT_DIR}/")